In [4]:
import pandas as pd
import numpy as np

np.random.seed(42)

print("🍷 СОЗДАЕМ ВИННЫЙ ДАТАСЕТ С ПРОПУСКАМИ")
print("=" * 60)

### 1. ЗАДАЕМ РАЗМЕРЫ КЛАССОВ (не идеально равные)
total = 601
# Пусть будет небольшой разброс
n_poor = 191     # плохих
n_medium = 215    # средних (чуть больше)
n_good = 195      # хороших (чуть меньше)

print(f"\n📊 ПЛАНИРУЕМОЕ РАСПРЕДЕЛЕНИЕ:")
print(f"Плохое вино:  {n_poor} строк ({n_poor/total*100:.1f}%)")
print(f"Среднее вино: {n_medium} строк ({n_medium/total*100:.1f}%)")
print(f"Хорошее вино: {n_good} строк ({n_good/total*100:.1f}%)")
print(f"Всего: {n_poor + n_medium + n_good} строк")

### 2. ФУНКЦИЯ ДЛЯ ДОБАВЛЕНИЯ ПРОПУСКОВ
def add_missing_values(df, missing_rate=0.03):
    """Добавляет случайные пропуски в датасет"""
    df_with_nan = df.copy()
    for col in df_with_nan.select_dtypes(include=[np.number]).columns:
        # Случайно выбираем индексы для пропусков
        n_missing = int(len(df_with_nan) * missing_rate * np.random.uniform(0.5, 1.5))
        missing_idx = np.random.choice(df_with_nan.index, n_missing, replace=False)
        df_with_nan.loc[missing_idx, col] = np.nan
    return df_with_nan

### 3. ГЕНЕРИРУЕМ ДАННЫЕ ДЛЯ КАЖДОГО КЛАССА

# ПЛОХОЕ ВИНО (quality 3-4) - 190 строк
poor_wine = pd.DataFrame({
    'fixed_acidity': np.random.normal(7.2, 0.7, n_poor),
    'volatile_acidity': np.random.normal(0.8, 0.2, n_poor),
    'citric_acid': np.random.normal(0.15, 0.08, n_poor),
    'residual_sugar': np.random.normal(1.8, 0.4, n_poor),
    'chlorides': np.random.normal(0.095, 0.02, n_poor),
    'free_sulfur_dioxide': np.random.normal(12, 4, n_poor),
    'total_sulfur_dioxide': np.random.normal(30, 8, n_poor),
    'density': np.random.normal(0.998, 0.001, n_poor),
    'pH': np.random.normal(3.4, 0.15, n_poor),
    'sulphates': np.random.normal(0.45, 0.1, n_poor),
    'alcohol': np.random.normal(9.0, 0.7, n_poor),
    'quality': np.random.choice([3, 4], n_poor),
    'quality_class': 'плохое'
})

# СРЕДНЕЕ ВИНО (quality 5-6) - 215 строк
medium_wine = pd.DataFrame({
    'fixed_acidity': np.random.normal(8.3, 0.8, n_medium),
    'volatile_acidity': np.random.normal(0.5, 0.15, n_medium),
    'citric_acid': np.random.normal(0.35, 0.15, n_medium),
    'residual_sugar': np.random.normal(2.5, 0.8, n_medium),
    'chlorides': np.random.normal(0.08, 0.015, n_medium),
    'free_sulfur_dioxide': np.random.normal(18, 6, n_medium),
    'total_sulfur_dioxide': np.random.normal(45, 12, n_medium),
    'density': np.random.normal(0.996, 0.0015, n_medium),
    'pH': np.random.normal(3.3, 0.15, n_medium),
    'sulphates': np.random.normal(0.6, 0.12, n_medium),
    'alcohol': np.random.normal(10.5, 1.0, n_medium),
    'quality': np.random.choice([5, 6], n_medium),
    'quality_class': 'среднее'
})

# ХОРОШЕЕ ВИНО (quality 7-8) - 195 строк
good_wine = pd.DataFrame({
    'fixed_acidity': np.random.normal(9.2, 0.9, n_good),
    'volatile_acidity': np.random.normal(0.3, 0.1, n_good),
    'citric_acid': np.random.normal(0.55, 0.15, n_good),
    'residual_sugar': np.random.normal(3.2, 1.0, n_good),
    'chlorides': np.random.normal(0.065, 0.01, n_good),
    'free_sulfur_dioxide': np.random.normal(25, 8, n_good),
    'total_sulfur_dioxide': np.random.normal(60, 15, n_good),
    'density': np.random.normal(0.994, 0.0015, n_good),
    'pH': np.random.normal(3.2, 0.15, n_good),
    'sulphates': np.random.normal(0.75, 0.15, n_good),
    'alcohol': np.random.normal(12.0, 1.2, n_good),
    'quality': np.random.choice([7, 8], n_good),
    'quality_class': 'хорошее'
})

### 4. ОБЪЕДИНЯЕМ ВСЕ КЛАССЫ
df_wine = pd.concat([poor_wine, medium_wine, good_wine], ignore_index=True)

# Перемешиваем строки (чтобы классы шли вперемешку)
df_wine = df_wine.sample(frac=1, random_state=42).reset_index(drop=True)

# Округляем числа для красоты
numeric_cols = df_wine.select_dtypes(include=[np.float64]).columns
df_wine[numeric_cols] = df_wine[numeric_cols].round(3)

print(f"\n✅ ДАТАСЕТ СОЗДАН:")
print(f"Форма: {df_wine.shape}")
print(f"Колонки: {df_wine.columns.tolist()}")

print(f"\n📊 ИТОГОВОЕ РАСПРЕДЕЛЕНИЕ КЛАССОВ:")
class_dist = df_wine['quality_class'].value_counts()
for cls in ['плохое', 'среднее', 'хорошее']:
    count = class_dist.get(cls, 0)
    print(f"{cls}: {count} строк ({count/len(df_wine)*100:.1f}%)")

### 5. ДОБАВЛЯЕМ ПРОПУСКИ (примерно 3-5% в случайных колонках)
print(f"\n🕳️ ДОБАВЛЯЕМ ПРОПУСКИ...")

# В разных колонках разное количество пропусков
for col in df_wine.select_dtypes(include=[np.number]).columns:
    if col != 'quality':  # quality оставим без пропусков (целевая)
        # Случайный процент пропусков от 2% до 7%
        missing_rate = np.random.uniform(0.02, 0.07)
        n_missing = int(len(df_wine) * missing_rate)
        missing_idx = np.random.choice(df_wine.index, n_missing, replace=False)
        df_wine.loc[missing_idx, col] = np.nan
        print(f"  {col}: {n_missing} пропусков ({missing_rate*100:.1f}%)")

### 6. ПРОВЕРЯЕМ РЕЗУЛЬТАТ
print(f"\n🔍 ПРОВЕРКА ДАТАСЕТА:")
print(f"Всего строк: {len(df_wine)}")
print(f"Всего колонок: {len(df_wine.columns)}")
print(f"\nПропуски в каждой колонке:")
missing = df_wine.isnull().sum()
print(missing[missing > 0])

print(f"\n📋 ПЕРВЫЕ 10 СТРОК:")
print(df_wine.head(10))

### 7. СОХРАНЯЕМ В ФАЙЛ
df_wine.to_csv('wine_dataset.csv', index=False)
print(f"\n💾 ДАТАСЕТ СОХРАНЕН: 'wine_dataset_600.csv'")

### 8. БЫСТРЫЙ EDA
print(f"\n📊 БЫСТРАЯ СТАТИСТИКА:")
print(df_wine.describe().round(2))

🍷 СОЗДАЕМ ВИННЫЙ ДАТАСЕТ С ПРОПУСКАМИ

📊 ПЛАНИРУЕМОЕ РАСПРЕДЕЛЕНИЕ:
Плохое вино:  191 строк (31.8%)
Среднее вино: 215 строк (35.8%)
Хорошее вино: 195 строк (32.4%)
Всего: 601 строк

✅ ДАТАСЕТ СОЗДАН:
Форма: (601, 13)
Колонки: ['fixed_acidity', 'volatile_acidity', 'citric_acid', 'residual_sugar', 'chlorides', 'free_sulfur_dioxide', 'total_sulfur_dioxide', 'density', 'pH', 'sulphates', 'alcohol', 'quality', 'quality_class']

📊 ИТОГОВОЕ РАСПРЕДЕЛЕНИЕ КЛАССОВ:
плохое: 191 строк (31.8%)
среднее: 215 строк (35.8%)
хорошее: 195 строк (32.4%)

🕳️ ДОБАВЛЯЕМ ПРОПУСКИ...
  fixed_acidity: 39 пропусков (6.5%)
  volatile_acidity: 30 пропусков (5.0%)
  citric_acid: 16 пропусков (2.7%)
  residual_sugar: 12 пропусков (2.1%)
  chlorides: 26 пропусков (4.5%)
  free_sulfur_dioxide: 15 пропусков (2.6%)
  total_sulfur_dioxide: 20 пропусков (3.4%)
  density: 33 пропусков (5.6%)
  pH: 27 пропусков (4.6%)
  sulphates: 15 пропусков (2.5%)
  alcohol: 40 пропусков (6.7%)

🔍 ПРОВЕРКА ДАТАСЕТА:
Всего строк: 601
Все